# Investment Research Agent — Week 1 Data Retrieval

This notebook completes the Week 1 data-retrieval tasks for the Investment Research Agent. It uses only sources permitted by the project scope:

- **Yahoo Finance (`yfinance`)** for historical prices, company information, and financial statements.
- **NewsAPI.org** for recent financial news.

The resulting data structures can later be passed to the news, earnings, and market-analysis agents.

## 1. Install dependencies

Run this cell once in the notebook environment.

In [ ]:
%pip install -q yfinance pandas requests

## 2. Import libraries

In [ ]:
from datetime import datetime, timedelta, timezone
from getpass import getpass
from typing import Optional

import pandas as pd
import requests
import yfinance as yf

## 3. Enter the NewsAPI key

Create a free API key at [NewsAPI.org](https://newsapi.org/). The secure prompt prevents the key from appearing in the notebook or its output. Do not commit an API key to GitHub.

In [ ]:
NEWS_API_KEY = getpass("Enter your NewsAPI key: ")

## 4. Data fields

The prototype retrieves the following fields:

| Category | Fields |
|---|---|
| Prices | ticker, date, open, high, low, close, adjusted close, volume, dividends, stock splits |
| Company | company name, sector, industry, country, website, currency, exchange, market capitalization, employees, business summary |
| Financials | income statement, balance sheet, and cash-flow statement |
| News | ticker, source, author, title, description, content, publication time, URL, search query |

## 5. Yahoo Finance retrieval functions

In [ ]:
def get_price_history(
    ticker: str,
    period: str = "1y",
    interval: str = "1d"
) -> pd.DataFrame:
    """Retrieve and standardize historical prices from Yahoo Finance."""
    symbol = ticker.strip().upper()
    prices = yf.Ticker(symbol).history(
        period=period,
        interval=interval,
        auto_adjust=False
    )

    if prices.empty:
        raise ValueError(f"No price data found for {symbol}.")

    prices = prices.reset_index().rename(
        columns={
            "Date": "date",
            "Datetime": "date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adjusted_close",
            "Volume": "volume",
            "Dividends": "dividends",
            "Stock Splits": "stock_splits"
        }
    )

    prices.insert(0, "ticker", symbol)

    selected_columns = [
        "ticker", "date", "open", "high", "low", "close",
        "adjusted_close", "volume", "dividends", "stock_splits"
    ]
    available_columns = [
        column for column in selected_columns if column in prices.columns
    ]

    return prices[available_columns]


def get_company_information(ticker: str) -> dict:
    """Retrieve selected company profile fields from Yahoo Finance."""
    symbol = ticker.strip().upper()
    info = yf.Ticker(symbol).get_info()

    if not info:
        raise ValueError(f"No company information found for {symbol}.")

    return {
        "ticker": symbol,
        "company_name": info.get("longName"),
        "sector": info.get("sector"),
        "industry": info.get("industry"),
        "country": info.get("country"),
        "website": info.get("website"),
        "currency": info.get("currency"),
        "exchange": info.get("exchange"),
        "market_cap": info.get("marketCap"),
        "employees": info.get("fullTimeEmployees"),
        "business_summary": info.get("longBusinessSummary")
    }


def get_financial_statements(ticker: str) -> dict:
    """Retrieve annual financial statements from Yahoo Finance."""
    symbol = ticker.strip().upper()
    stock = yf.Ticker(symbol)

    income_statement = stock.get_income_stmt(freq="yearly")
    balance_sheet = stock.get_balance_sheet(freq="yearly")
    cash_flow = stock.get_cashflow(freq="yearly")

    if (
        income_statement.empty
        and balance_sheet.empty
        and cash_flow.empty
    ):
        raise ValueError(f"No financial statements found for {symbol}.")

    return {
        "income_statement": income_statement,
        "balance_sheet": balance_sheet,
        "cash_flow": cash_flow
    }

## 6. NewsAPI retrieval function

The search combines the company name with the ticker to reduce unrelated results. NewsAPI may provide only a shortened content preview, so the initial workflow primarily uses titles and descriptions.

In [ ]:
def get_company_news(
    ticker: str,
    company_name: str,
    api_key: str,
    days_back: int = 7,
    page_size: int = 20
) -> pd.DataFrame:
    """Retrieve recent English-language company news from NewsAPI."""
    if not api_key:
        raise ValueError("A NewsAPI key is required.")

    symbol = ticker.strip().upper()
    end_date = datetime.now(timezone.utc)
    start_date = end_date - timedelta(days=days_back)
    search_query = f'"{company_name}" OR "{symbol} stock"'

    parameters = {
        "q": search_query,
        "from": start_date.date().isoformat(),
        "to": end_date.date().isoformat(),
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": min(page_size, 100),
        "apiKey": api_key
    }

    response = requests.get(
        "https://newsapi.org/v2/everything",
        params=parameters,
        timeout=30
    )
    response.raise_for_status()
    response_data = response.json()

    if response_data.get("status") != "ok":
        message = response_data.get("message", "NewsAPI request failed.")
        raise RuntimeError(message)

    records = []
    for article in response_data.get("articles", []):
        records.append({
            "ticker": symbol,
            "source": article.get("source", {}).get("name"),
            "author": article.get("author"),
            "title": article.get("title"),
            "description": article.get("description"),
            "content": article.get("content"),
            "published_at": article.get("publishedAt"),
            "url": article.get("url"),
            "search_query": search_query
        })

    news = pd.DataFrame(records)

    if not news.empty:
        news["published_at"] = pd.to_datetime(
            news["published_at"],
            utc=True,
            errors="coerce"
        )
        news = news.drop_duplicates(subset=["url"]).reset_index(drop=True)

    return news

## 7. Combined data-collection function

In [ ]:
def collect_investment_data(
    ticker: str,
    news_api_key: Optional[str] = None,
    price_period: str = "1y",
    news_days: int = 7
) -> dict:
    """Collect company, price, financial, and news data for one stock."""
    symbol = ticker.strip().upper()
    company = get_company_information(symbol)
    company_name = company.get("company_name") or symbol

    return {
        "ticker": symbol,
        "company": company,
        "prices": get_price_history(symbol, period=price_period),
        "financials": get_financial_statements(symbol),
        "news": get_company_news(
            ticker=symbol,
            company_name=company_name,
            api_key=news_api_key,
            days_back=news_days
        )
    }

## 8. Test with Apple (`AAPL`)

Change `TEST_TICKER` to test another publicly traded company.

In [ ]:
TEST_TICKER = "AAPL"

investment_data = collect_investment_data(
    ticker=TEST_TICKER,
    news_api_key=NEWS_API_KEY,
    price_period="1y",
    news_days=7
)

company_df = pd.DataFrame([investment_data["company"]])
prices_df = investment_data["prices"]
income_statement = investment_data["financials"]["income_statement"]
balance_sheet = investment_data["financials"]["balance_sheet"]
cash_flow = investment_data["financials"]["cash_flow"]
news_df = investment_data["news"]

print(f"Company: {investment_data['company']['company_name']}")
print(f"Price records: {len(prices_df)}")
print(f"News articles: {len(news_df)}")

## 9. Display retrieved data

In [ ]:
display(company_df)
display(prices_df.head())
display(income_statement)
display(balance_sheet)
display(cash_flow)

if news_df.empty:
    print("No recent articles were returned for the selected date range.")
else:
    display(news_df[[
        "published_at", "source", "title", "description", "url"
    ]].head(10))

## 10. Validate the Week 1 prototype

These checks confirm that the core retrieval functions returned the fields needed by later agent workflows. A lack of recent news is reported as a warning rather than a failed retrieval test because article availability depends on the company and selected date range.

In [ ]:
required_price_fields = {
    "ticker", "date", "open", "high", "low", "close", "volume"
}
required_news_fields = {
    "ticker", "source", "title", "published_at", "url"
}

assert investment_data["company"]["ticker"] == TEST_TICKER
assert investment_data["company"]["company_name"] is not None
assert not prices_df.empty
assert required_price_fields.issubset(prices_df.columns)
assert not income_statement.empty
assert not balance_sheet.empty
assert not cash_flow.empty

if news_df.empty:
    print("Warning: NewsAPI returned no recent articles.")
else:
    assert required_news_fields.issubset(news_df.columns)

print("All available Week 1 retrieval tests passed successfully.")

## Conclusion

The Week 1 prototype accepts a stock symbol and retrieves structured company, price, financial-statement, and news data. These outputs establish the tool layer needed for later planning, prompt chaining, specialist routing, self-reflection, and evaluator–optimizer workflows.